# 15.12 Expert 容量不夠怎麼辦？

# 第 15 章：從 Dense 到 Mixture of Experts（MoE）

前置：第4章Dense FFN。把token想成訂單，experts像不同廚師，router像分單員。這個譬喻只描述分工：專家不會天生知道中文或數學，專長需由資料與訓練觀察。實作是dropless小型MoE，不需分散式系統。




In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：送出時分組，收回時依原訂單編號**

同token多份貢獻要相加，不能覆蓋。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/dispatch.svg")))

**先想一想：**容量2而有5單，沒處理的3單該怎麼記錄？

先讓所有訂單都送到選中專家，再考慮每位廚師容量上限。超過上限要明確選擇drop、fallback或再分派，不能默默丟token。

**把直覺寫成數學：**capacity≈ceil(capacity_factor·tokens·k/experts)。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
chosen = torch.tensor([0, 0, 0, 1, 0, 0])
capacity = 2
counts = torch.bincount(chosen, minlength=2)
overflow = (counts - capacity).clamp(min=0)
print(counts.tolist(), "overflow", overflow.tolist())
print("本模型dropless；容量限制為另設policy的選修")

**如何讀結果：**這個數值檢查沒有改變主模型；實現capacity版本時測輸出與梯度，不只是load。

**只改一件事：**只改capacity，保持router選擇。
